In [1]:
import pandas as pd

# Load the dataset again
df = pd.read_csv(r'D:\twitter-sentiment-analyzer\data\training.1600000.processed.noemoticon.csv',
                 encoding='latin-1',
                 names=['target','id','date','flag','user','text'])

# Keep only what we need and convert target
df = df[['text', 'target']]
df['target'] = df['target'].map({0: 0, 4: 1})

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (1600000, 2)


In [2]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download required NLTK data
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

print("NLTK data downloaded successfully!")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\celes\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\celes\AppData\Roaming\nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\celes\AppData\Roaming\nltk_data...


NLTK data downloaded successfully!


In [3]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_tweet(text):
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove @mentions
    text = re.sub(r'@\w+', '', text)
    # Remove #hashtags (keep the word, remove the #)
    text = re.sub(r'#', '', text)
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Lowercase
    text = text.lower()
    # Tokenize, lemmatize, remove stopwords
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words and len(w) > 2]
    return ' '.join(words)

# Test it on a sample tweet first
sample = "@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer. You shoulda got David Carr of Third Day to do it ;D"
print("Original:", sample)
print("Cleaned: ", clean_tweet(sample))

Original: @switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer. You shoulda got David Carr of Third Day to do it ;D
Cleaned:  thats bummer shoulda got david carr third day


In [4]:
# Use a sample of 200,000 tweets for faster processing
df_sample = df.sample(200000, random_state=42)

print("Cleaning tweets... this will take 3-5 minutes, please wait!")
df_sample['cleaned_text'] = df_sample['text'].apply(clean_tweet)

# Remove any empty rows after cleaning
df_sample = df_sample[df_sample['cleaned_text'].str.strip() != '']

print(f"Done! Cleaned {len(df_sample)} tweets")
print("\nSample comparison:")
print(df_sample[['text', 'cleaned_text']].head(3))

Cleaning tweets... this will take 3-5 minutes, please wait!
Done! Cleaned 198790 tweets

Sample comparison:
                                                     text  \
541200             @chrishasboobs AHHH I HOPE YOUR OK!!!    
750     @misstoriblack cool , i have no tweet apps  fo...   
766711  @TiannaChaos i know  just family drama. its la...   

                                             cleaned_text  
541200                                          ahhh hope  
750                                  cool tweet apps razr  
766711  know family drama lamehey next time hang kim g...  


In [5]:
# Save cleaned data to the data folder
df_sample.to_csv(r'D:\twitter-sentiment-analyzer\data\cleaned_tweets.csv', index=False)

print("Cleaned data saved successfully!")
print(f"Total tweets saved: {len(df_sample)}")
print(f"Positive tweets: {df_sample[df_sample['target']==1]['target'].count()}")
print(f"Negative tweets: {df_sample[df_sample['target']==0]['target'].count()}")

Cleaned data saved successfully!
Total tweets saved: 198790
Positive tweets: 99487
Negative tweets: 99303
